# Sequence Windows

**Owner:** Prateek Shakya

## Purpose
Demonstrate and freeze the causal, per-engine sliding-window pipeline used by all Phase 2 models.

## Method
The scaler is fitted on the 80 training engines only. Windows are built separately per engine, left-padded by repeating the first observation, and optionally selected at fixed validation or test points.

In [ ]:
from pathlib import Path
import sys
import subprocess
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "src"))
from load_data import load_train, load_test
from evaluation import split_train_val, validation_points, test_points
from sequences import fit_scaler, make_windows, DEFAULT_WINDOW

train, test = load_train(), load_test()
train_fit, val = split_train_val(train)
scaler = fit_scaler(train_fit)
X_train, y_train, _ = make_windows(train_fit, scaler, DEFAULT_WINDOW)
X_val, y_val, _ = make_windows(val, scaler, DEFAULT_WINDOW)
X_points, y_points, point_keys = make_windows(val, scaler, DEFAULT_WINDOW, validation_points(val))
X_test, y_test, test_keys = make_windows(test, scaler, DEFAULT_WINDOW, test_points(test))
print({"train": X_train.shape, "val": X_val.shape, "validation_points": X_points.shape, "test_points": X_test.shape, "default_window": DEFAULT_WINDOW})


### Window counts and memory
Padding keeps one sample per observed row, so changing the window length changes memory and context but not the number of training windows.

In [ ]:
for w in [10,20,30,50]:
    X, _, _ = make_windows(train_fit, scaler, w)
    print(w, X.shape, f"{X.nbytes / 1024**2:.2f} MiB")
X50, _, _ = make_windows(train_fit, scaler, 50)
print("window-50 memory formula: rows * 50 * 14 * 4 bytes =", X50.nbytes, "bytes")


### Inspect one causal window
The heat map shows the 30 time steps and 14 scaled sensors for one validation point. The final column in time is the prediction cycle; no future row is present.

In [ ]:
plt.figure(figsize=(9,4)); plt.imshow(X_points[0].T, aspect="auto", cmap="coolwarm"); plt.colorbar(label="scaled value"); plt.xlabel("Window position (old to current)"); plt.ylabel("Sensor index"); plt.title("One scaled causal window"); plt.tight_layout(); plt.show()


### Show left padding
For a short history, the first observed vector is repeated on the left. This is deterministic and does not use a future observation.

In [ ]:
short_id = int(test.groupby("engine_id").size().idxmin())
short = test[test.engine_id == short_id]
short_X, _, _ = make_windows(short, scaler, 50)
print("short test engine", short_id, "cycles", len(short), "window", short_X.shape)
plt.figure(figsize=(9,3)); plt.plot(short_X[0,:,0], marker="."); plt.axvline(48-len(short), color="red", linestyle="--", label="observed history starts"); plt.xlabel("Window position"); plt.ylabel("Scaled sensor 2"); plt.legend(); plt.title("Left padding in the first short-engine window"); plt.tight_layout(); plt.show()


### Automated checks
The repository test suite verifies output shape, causal indexing, engine isolation, left padding, point ordering, missing points, scaler scope, targets, constants, and unlabeled test data.

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=ROOT, capture_output=True, text=True)
print(result.stdout)
if result.returncode:
    raise RuntimeError(result.stderr)


## Results
The printed shapes, window-count table, memory calculation, plots, and test output are the reproducibility record for the frozen pipeline.

## Conclusion
The default contract is one causal `(30, 14)` window per observed row, with training-only normalization and explicit ordered point selection. Later models can change the window length or opt into target capping without changing the public API.

## Handoff
Use `fit_scaler`, `make_windows`, `DEFAULT_WINDOW`, and optionally `cap_rul` from `src/sequences.py`. Validation-point order matches `validation_points`; test-point order matches `test_points`.